In [0]:
# =========================================
# US09 - Bike Share Preprocessing (Bronze -> Silver)
# Databricks Serverless / UC Volume safe
# Granularity: hour ("HH:00"), floor for both start and end time
# =========================================

from pyspark.sql import functions as F
import re

# ---------------------------
# Paths (adjust only if your team changes the project folder)
# ---------------------------
BRONZE_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze/bikeshare_ridership"
SILVER_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/bikeshare_trips"

# Business rule: remove unrealistic trips (operational anomalies)
MAX_DURATION_MIN = 240  # 4 hours

print("=== Bike Share Preprocessing ===")
print("BRONZE:", BRONZE_DIR)
print("SILVER:", SILVER_DIR)

# ---------------------------
# 1) Load Bronze (fresh read to avoid carrying old transformations)
# ---------------------------
df_bronze = spark.read.parquet(BRONZE_DIR)

print("Bronze schema (original):")
df_bronze.printSchema()

# ---------------------------
# 2) Standardize column names to snake_case
#    Example: "Trip  Duration" -> "trip_duration"
# ---------------------------
def to_snake(name: str) -> str:
    s = name.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)   # replace spaces/symbols by underscore
    s = re.sub(r"_+", "_", s).strip("_")
    return s

df = df_bronze
for c in df_bronze.columns:
    df = df.withColumnRenamed(c, to_snake(c))

print("Columns after rename:")
print(df.columns)

# Ensure start/end times are treated as strings (avoid implicit timestamp casting)
df = (df
      .withColumn("start_time", F.col("start_time").cast("string"))
      .withColumn("end_time",   F.col("end_time").cast("string"))
      .withColumn("trip_duration", F.col("trip_duration").cast("string"))
)

# ---------------------------
# 3) Basic null checks (keep only rows needed for modeling/integration)
# ---------------------------
df = df.filter(
    F.col("trip_id").isNotNull() &
    F.col("start_time").isNotNull() &
    F.col("end_time").isNotNull() &
    F.col("start_station_id").isNotNull() &
    F.col("end_station_id").isNotNull() &
    F.col("year").isNotNull() &
    F.col("month").isNotNull()
)

# ---------------------------
# 4) Extract day + hour buckets WITHOUT timestamps (serverless-safe)
#    Start/End formats observed:
#      "7/1/2024 0:00"
#      "07/01/2023 00:00"
#    We parse by string splits (no to_timestamp / unix_timestamp).
# ---------------------------
df = (df
    .withColumn("start_date_part", F.split(F.col("start_time"), " ").getItem(0))
    .withColumn("start_time_part", F.split(F.col("start_time"), " ").getItem(1))
    .withColumn("end_date_part",   F.split(F.col("end_time"), " ").getItem(0))
    .withColumn("end_time_part",   F.split(F.col("end_time"), " ").getItem(1))

    # day from M/d/yyyy -> second element (index 1)
    .withColumn("day", F.split(F.col("start_date_part"), "/").getItem(1).cast("int"))

    # hour from H:mm -> left side (index 0)
    .withColumn("start_hour", F.split(F.col("start_time_part"), ":").getItem(0).cast("int"))
    .withColumn("end_hour",   F.split(F.col("end_time_part"), ":").getItem(0).cast("int"))

    # hour string format HH:00 (granularity: hour)
    .withColumn("hour_start_str", F.format_string("%02d:00", F.col("start_hour")))
    .withColumn("hour_end_str",   F.format_string("%02d:00", F.col("end_hour")))
)

# Sanity filters on extracted fields
df = df.filter(
    F.col("day").between(1, 31) &
    F.col("start_hour").between(0, 23) &
    F.col("end_hour").between(0, 23)
)

# ---------------------------
# 5) Parse duration (seconds -> minutes) + filter outliers
#    Based on your summary: median ~683, max huge => seconds
# ---------------------------
df = df.withColumn(
    "trip_duration_num",
    F.regexp_extract(F.col("trip_duration"), r"([0-9]+(\.[0-9]+)?)", 1).cast("double")
)

df = df.filter(F.col("trip_duration_num").isNotNull())

# Convert seconds -> minutes
df = df.withColumn("trip_duration_min", F.col("trip_duration_num") / 60.0)

# Remove invalid or extreme durations
df = df.filter(
    (F.col("trip_duration_min") > 0) &
    (F.col("trip_duration_min") <= MAX_DURATION_MIN)
)

# ---------------------------
# 6) De-duplicate by trip_id (keep first occurrence)
# ---------------------------
df = df.dropDuplicates(["trip_id"])

# ---------------------------
# 7) Build Silver dataset (clean, standardized, modeling-friendly)
# ---------------------------
df_silver = df.select(
    "trip_id",
    "bike_id",
    "user_type",
    "model",
    "start_station_id",
    "start_station_name",
    "end_station_id",
    "end_station_name",
    "year", "month", "day",
    "hour_start_str",
    "hour_end_str",
    "trip_duration_min",
    "source_file"
)

# ---------------------------
# 8) Write Silver Parquet (partitioned by year/month)
# ---------------------------
(df_silver.write
 .mode("overwrite")
 .partitionBy("year", "month")
 .parquet(SILVER_DIR)
)

print("✅ Silver written to:", SILVER_DIR)

# ---------------------------
# 9) Validations (evidence for Taiga/PR)
# ---------------------------
total_rows = df_silver.count()
print(f"Total rows in Silver: {total_rows:,}")

print("=== Duration summary (minutes) ===")
display(
    df_silver.select("trip_duration_min")
             .summary("count", "min", "25%", "50%", "75%", "max", "mean")
)

print("=== Duration buckets ===")
display(
    df_silver.groupBy(
        F.when(F.col("trip_duration_min") <= 60, "0–1h")
         .when(F.col("trip_duration_min") <= 120, "1–2h")
         .when(F.col("trip_duration_min") <= 240, "2–4h")
         .otherwise(">4h")
         .alias("duration_bucket")
    ).count().orderBy("duration_bucket")
)

print("=== Rows per year ===")
display(df_silver.groupBy("year").count().orderBy("year"))

print("=== Rows per year & month ===")
display(df_silver.groupBy("year", "month").count().orderBy("year", "month"))

print("=== Silver Schema ===")
df_silver.printSchema()

print("=== DONE (US09 Bike Share preprocessing) ===")

# Downstream example:
# df_silver_read = spark.read.parquet(SILVER_DIR)


=== Bike Share Preprocessing ===
BRONZE: dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze/bikeshare_ridership
SILVER: dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/bikeshare_trips
Bronze schema (original):
root
 |-- Trip Id: string (nullable = true)
 |-- Trip  Duration: string (nullable = true)
 |-- Start Station Id: string (nullable = true)
 |-- Start Time: string (nullable = true)
 |-- Start Station Name: string (nullable = true)
 |-- End Station Id: string (nullable = true)
 |-- End Time: string (nullable = true)
 |-- End Station Name: string (nullable = true)
 |-- Bike Id: string (nullable = true)
 |-- User Type: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

Columns after rename:
['trip_id', 'trip_duration', 'start_station_id', 'start_time', 'start_station_name', 'end_station_id', 'end_time', 'end_station_name'

summary,trip_duration_min
count,12028835
min,0.03333333333333333
25%,7.0
50%,11.383333333333333
75%,18.4
max,240.0
mean,15.370950439784101


=== Duration buckets ===


duration_bucket,count
0–1h,11788549
1–2h,198533
2–4h,41753


=== Rows per year ===


year,count
2022,998010
2023,5697944
2024,5332881


=== Rows per year & month ===


year,month,count
2022,10,498749
2022,11,319582
2022,12,179679
2023,1,179690
2023,2,172416
2023,3,224092
2023,4,379463
2023,5,587567
2023,6,661717
2023,7,733730


=== Silver Schema ===
root
 |-- trip_id: string (nullable = true)
 |-- bike_id: string (nullable = true)
 |-- user_type: string (nullable = true)
 |-- model: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour_start_str: string (nullable = false)
 |-- hour_end_str: string (nullable = false)
 |-- trip_duration_min: double (nullable = true)
 |-- source_file: string (nullable = true)

=== DONE (US09 Bike Share preprocessing) ===
